# NREL EVI-Pro Lite API Demo

This notebook demonstrates how to connect to NREL's EVI-Pro Lite API and explore electric vehicle charging load profiles.

## Overview
The EVI-Pro Lite API provides output from NREL's EVI-Pro model and offers daily (24-hour) fleet-level charging load profiles for customizable electric vehicle scenarios.

**API Documentation**: https://developer.nrel.gov/docs/transportation/evi-pro-lite-v1/  
**GitHub Repository**: https://github.com/NREL/EVI-Pro-Lite

## Requirements
- NREL API Key (sign up at https://developer.nrel.gov/signup/)
- Python packages: requests, pandas, numpy, matplotlib, seaborn

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from datetime import datetime
import os
from itertools import product
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("📦 All packages imported successfully!")

## 🔑 API Key Setup

You need an NREL API key to access the EVI-Pro Lite API. 

**Option 1**: Get your free API key at https://developer.nrel.gov/signup/  
**Option 2**: Use the demo key (rate limited) for testing

In [ ]:
# Option 1: Use your API key (recommended)
# API_KEY = "YOUR_API_KEY_HERE"  # Replace with your actual API key

# Option 2: Use demo key (rate limited)
API_KEY = "DEMO_KEY"

print(f"✅ Using API key: {API_KEY[:10]}..." if API_KEY != "DEMO_KEY" else "⚠️  Using DEMO_KEY (rate limited)")

## 🛣️ EVI-Pro Lite API Endpoints

The main endpoint for the EVI-Pro Lite API is:
`https://developer.nrel.gov/api/evi-pro-lite/v1/daily-load-profile`

This endpoint provides daily charging load profiles based on various fleet and charging behavior parameters.

In [ ]:
# Base API endpoint
BASE_URL = "https://developer.nrel.gov/api/evi-pro-lite/v1/daily-load-profile"

def make_api_request(api_key, **params):
    """
    Make a request to the EVI-Pro Lite API
    
    Args:
        api_key: NREL API key
        **params: API parameters
    
    Returns:
        dict: JSON response from API
    """
    # Add API key to parameters
    params['api_key'] = api_key
    
    try:
        response = requests.get(BASE_URL, params=params, timeout=30)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"❌ API request failed: {e}")
        return None
    except json.JSONDecodeError as e:
        print(f"❌ Failed to parse JSON response: {e}")
        return None

print("🔧 API request function defined")

## 📊 API Parameters Reference

The EVI-Pro Lite API accepts the following parameters to customize charging scenarios:

In [ ]:
# Complete parameter options for EVI-Pro Lite API
API_PARAMETERS = {
    'fleet_size': {
        'description': 'Number of vehicles in the fleet',
        'options': [50000, 10000, 1000],
        'default': 10000
    },
    'mean_dvmt': {
        'description': 'Mean daily vehicle miles traveled',
        'options': [45, 35, 25],
        'default': 35
    },
    'temp_c': {
        'description': 'Temperature in Celsius',
        'options': [40, 30, 20, 10, 0, -10, -20],
        'default': 20
    },
    'pev_type': {
        'description': 'Plug-in Electric Vehicle type',
        'options': ['PHEV20', 'PHEV50', 'BEV100', 'BEV250'],
        'default': 'BEV250'
    },
    'pev_dist': {
        'description': 'PEV distribution in fleet',
        'options': ['BEV', 'PHEV', 'EQUAL'],
        'default': 'BEV'
    },
    'class_dist': {
        'description': 'Vehicle class distribution',
        'options': ['Sedan', 'SUV', 'Equal'],
        'default': 'Equal'
    },
    'home_access_dist': {
        'description': 'Home charging access distribution',
        'options': ['HA100', 'HA75', 'HA50'],
        'default': 'HA75'
    },
    'home_power_dist': {
        'description': 'Home charging power distribution',
        'options': ['MostL1', 'MostL2', 'Equal'],
        'default': 'MostL2'
    },
    'work_power_dist': {
        'description': 'Work charging power distribution',
        'options': ['MostL1', 'MostL2', 'Equal'],
        'default': 'MostL2'
    },
    'pref_dist': {
        'description': 'Charging preference distribution',
        'options': ['Home60', 'Home80', 'Home100'],
        'default': 'Home80'
    },
    'res_charging': {
        'description': 'Residential charging behavior',
        'options': ['min_delay', 'max_delay', 'midnight_charge'],
        'default': 'min_delay'
    },
    'work_charging': {
        'description': 'Work charging behavior',
        'options': ['min_delay', 'max_delay'],
        'default': 'min_delay'
    }
}

# Display parameter reference
print("📋 EVI-Pro Lite API Parameters:")
print("=" * 50)
for param, info in API_PARAMETERS.items():
    print(f"\n🔸 {param}")
    print(f"   Description: {info['description']}")
    print(f"   Options: {info['options']}")
    print(f"   Default: {info['default']}")

## 🧪 Basic API Test

Let's make a basic API call with default parameters to test connectivity:

In [ ]:
# Test API connectivity with default parameters
default_params = {param: info['default'] for param, info in API_PARAMETERS.items()}

print("🔄 Testing API with default parameters...")
print(f"Parameters: {default_params}")

test_response = make_api_request(API_KEY, **default_params)

if test_response:
    print("\n✅ API connection successful!")
    print(f"Response keys: {list(test_response.keys())}")
    
    # Check if we have load profile data
    if 'outputs' in test_response:
        outputs = test_response['outputs']
        print(f"\n📊 Output data keys: {list(outputs.keys())}")
        
        # Display sample of load profile data
        if 'load_profile_kw' in outputs:
            load_profile = outputs['load_profile_kw']
            print(f"\n⚡ Load profile data points: {len(load_profile)}")
            print(f"Sample load values (first 6 hours): {load_profile[:6]}")
    else:
        print(f"\n📄 Full response: {test_response}")
else:
    print("❌ API test failed")

## 📈 Visualize Basic Load Profile

Let's create a visualization of the charging load profile from our test API call:

In [ ]:
def plot_load_profile(api_response, title="EV Charging Load Profile"):
    """
    Plot the charging load profile from API response
    
    Args:
        api_response: JSON response from EVI-Pro Lite API
        title: Plot title
    """
    if not api_response or 'outputs' not in api_response:
        print("❌ No valid data to plot")
        return
    
    outputs = api_response['outputs']
    if 'load_profile_kw' not in outputs:
        print("❌ No load profile data in response")
        return
    
    load_profile = outputs['load_profile_kw']
    hours = list(range(24))  # 24-hour profile
    
    # Create the plot
    plt.figure(figsize=(12, 6))
    plt.plot(hours, load_profile, marker='o', linewidth=2, markersize=4)
    plt.title(title, fontsize=16, fontweight='bold')
    plt.xlabel('Hour of Day', fontsize=12)
    plt.ylabel('Charging Load (kW)', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.xticks(range(0, 24, 2))
    
    # Add statistics
    max_load = max(load_profile)
    max_hour = load_profile.index(max_load)
    total_energy = sum(load_profile)
    
    plt.axvline(x=max_hour, color='red', linestyle='--', alpha=0.7, label=f'Peak at {max_hour}:00')
    plt.legend()
    
    # Add text box with statistics
    stats_text = f'Peak Load: {max_load:.1f} kW\nPeak Hour: {max_hour}:00\nTotal Energy: {total_energy:.1f} kWh'
    plt.text(0.02, 0.98, stats_text, transform=plt.gca().transAxes, 
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
    
    plt.tight_layout()
    plt.show()
    
    print(f"📊 Peak charging load: {max_load:.1f} kW at {max_hour}:00")
    print(f"⚡ Total daily energy: {total_energy:.1f} kWh")

# Plot the test response
if test_response:
    plot_load_profile(test_response, "Basic EV Charging Load Profile (Default Parameters)")
else:
    print("⚠️ No test response available to plot")

## 🔬 Parameter Exploration

Let's explore how different parameters affect the charging load profile:

In [ ]:
def compare_parameters(base_params, param_to_vary, values_to_test, api_key):
    """
    Compare load profiles by varying a single parameter
    
    Args:
        base_params: Base parameter set
        param_to_vary: Parameter name to vary
        values_to_test: List of values to test for the parameter
        api_key: NREL API key
    """
    plt.figure(figsize=(14, 8))
    hours = list(range(24))
    
    results = {}
    
    for value in values_to_test:
        # Create parameters with varied value
        test_params = base_params.copy()
        test_params[param_to_vary] = value
        
        print(f"🔄 Testing {param_to_vary} = {value}...")
        response = make_api_request(api_key, **test_params)
        
        if response and 'outputs' in response and 'load_profile_kw' in response['outputs']:
            load_profile = response['outputs']['load_profile_kw']
            results[value] = load_profile
            
            # Plot this profile
            plt.plot(hours, load_profile, marker='o', linewidth=2, 
                    markersize=3, label=f'{param_to_vary}={value}')
        else:
            print(f"❌ Failed to get data for {param_to_vary} = {value}")
    
    if results:
        plt.title(f'EV Charging Load Profiles - Varying {param_to_vary}', fontsize=16, fontweight='bold')
        plt.xlabel('Hour of Day', fontsize=12)
        plt.ylabel('Charging Load (kW)', fontsize=12)
        plt.grid(True, alpha=0.3)
        plt.xticks(range(0, 24, 2))
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()
        
        # Print summary statistics
        print(f"\n📊 Summary for {param_to_vary} comparison:")
        for value, profile in results.items():
            peak_load = max(profile)
            total_energy = sum(profile)
            print(f"  {value}: Peak = {peak_load:.1f} kW, Total = {total_energy:.1f} kWh")
    else:
        print(f"❌ No successful API calls for {param_to_vary} comparison")
    
    return results

print("🔧 Parameter comparison function defined")

### 🌡️ Temperature Impact Analysis

In [ ]:
# Test different temperatures
temp_results = compare_parameters(
    base_params=default_params,
    param_to_vary='temp_c',
    values_to_test=[-10, 0, 20, 40],  # Cold to hot temperatures
    api_key=API_KEY
)

### 🚗 Vehicle Type Impact Analysis

In [ ]:
# Test different PEV types
pev_results = compare_parameters(
    base_params=default_params,
    param_to_vary='pev_type',
    values_to_test=['PHEV20', 'PHEV50', 'BEV100', 'BEV250'],
    api_key=API_KEY
)

### 🏠 Charging Behavior Impact Analysis

In [ ]:
# Test different residential charging behaviors
charging_results = compare_parameters(
    base_params=default_params,
    param_to_vary='res_charging',
    values_to_test=['min_delay', 'max_delay', 'midnight_charge'],
    api_key=API_KEY
)

## 🎯 California-Specific Analysis

Let's create scenarios relevant to California's climate and EV adoption patterns:

In [ ]:
# Define California-specific scenarios
california_scenarios = {
    'Bay Area Summer': {
        'temp_c': 20,
        'pev_type': 'BEV250',
        'pev_dist': 'BEV',
        'home_access_dist': 'HA75',
        'res_charging': 'max_delay',  # Time of use rates encourage delayed charging
        'mean_dvmt': 35
    },
    'LA Basin Winter': {
        'temp_c': 10,
        'pev_type': 'BEV250',
        'pev_dist': 'BEV',
        'home_access_dist': 'HA50',  # Lower home access due to apartments
        'res_charging': 'min_delay',
        'mean_dvmt': 45  # Higher driving in LA
    },
    'Central Valley Summer': {
        'temp_c': 40,  # Very hot summers
        'pev_type': 'BEV100',
        'pev_dist': 'EQUAL',
        'home_access_dist': 'HA100',  # More single-family homes
        'res_charging': 'midnight_charge',  # Avoid peak demand
        'mean_dvmt': 45
    }
}

# Run California scenarios
print("🌴 Running California-specific EV charging scenarios...")

ca_results = {}
plt.figure(figsize=(15, 10))

for i, (scenario_name, scenario_params) in enumerate(california_scenarios.items()):
    # Merge with default params
    full_params = default_params.copy()
    full_params.update(scenario_params)
    
    print(f"\n🔄 Running scenario: {scenario_name}")
    print(f"   Key parameters: {scenario_params}")
    
    response = make_api_request(API_KEY, **full_params)
    
    if response and 'outputs' in response and 'load_profile_kw' in response['outputs']:
        load_profile = response['outputs']['load_profile_kw']
        ca_results[scenario_name] = {
            'profile': load_profile,
            'params': scenario_params
        }
        
        # Create subplot for each scenario
        plt.subplot(2, 2, i+1)
        hours = list(range(24))
        plt.plot(hours, load_profile, marker='o', linewidth=2, markersize=3)
        plt.title(f'{scenario_name}\n(Peak: {max(load_profile):.1f} kW)', fontsize=12, fontweight='bold')
        plt.xlabel('Hour of Day')
        plt.ylabel('Load (kW)')
        plt.grid(True, alpha=0.3)
        plt.xticks(range(0, 24, 4))
        
        print(f"   ✅ Peak load: {max(load_profile):.1f} kW")
        print(f"   ⚡ Total energy: {sum(load_profile):.1f} kWh")
    else:
        print(f"   ❌ Failed to get data for {scenario_name}")

# Add overall comparison plot
plt.subplot(2, 2, 4)
hours = list(range(24))
for scenario_name, data in ca_results.items():
    plt.plot(hours, data['profile'], linewidth=2, label=scenario_name)

plt.title('California Scenarios Comparison', fontsize=12, fontweight='bold')
plt.xlabel('Hour of Day')
plt.ylabel('Load (kW)')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=8)
plt.xticks(range(0, 24, 4))

plt.tight_layout()
plt.show()

print(f"\n📊 Completed {len(ca_results)} California scenarios")

## 📊 Comprehensive Parameter Sensitivity Analysis

Let's create a systematic analysis of how each parameter affects peak load and total energy:

In [ ]:
def sensitivity_analysis(api_key, max_requests=20):
    """
    Perform sensitivity analysis on key parameters
    
    Args:
        api_key: NREL API key
        max_requests: Maximum number of API requests to make
    """
    # Select key parameters for sensitivity analysis
    sensitivity_params = {
        'temp_c': [0, 20, 40],
        'pev_type': ['BEV100', 'BEV250'],
        'res_charging': ['min_delay', 'max_delay'],
        'home_access_dist': ['HA50', 'HA100']
    }
    
    results = []
    request_count = 0
    
    print(f"🔬 Starting sensitivity analysis (max {max_requests} requests)...")
    
    # Generate all combinations of parameters
    param_names = list(sensitivity_params.keys())
    param_values = list(sensitivity_params.values())
    
    for combination in product(*param_values):
        if request_count >= max_requests:
            print(f"⚠️ Reached maximum requests limit ({max_requests})")
            break
            
        # Create parameter set
        test_params = default_params.copy()
        for i, param_name in enumerate(param_names):
            test_params[param_name] = combination[i]
        
        print(f"🔄 Request {request_count + 1}: {dict(zip(param_names, combination))}")
        
        response = make_api_request(api_key, **test_params)
        request_count += 1
        
        if response and 'outputs' in response and 'load_profile_kw' in response['outputs']:
            profile = response['outputs']['load_profile_kw']
            
            result = {
                'peak_load': max(profile),
                'total_energy': sum(profile),
                'peak_hour': profile.index(max(profile))
            }
            
            # Add parameter values
            for i, param_name in enumerate(param_names):
                result[param_name] = combination[i]
            
            results.append(result)
            print(f"   ✅ Peak: {result['peak_load']:.1f} kW, Total: {result['total_energy']:.1f} kWh")
        else:
            print(f"   ❌ Request failed")
    
    # Convert to DataFrame for analysis
    if results:
        df = pd.DataFrame(results)
        print(f"\n📊 Collected {len(results)} successful responses")
        
        # Create visualization
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Peak load by temperature
        axes[0, 0].boxplot([df[df['temp_c'] == temp]['peak_load'] for temp in sensitivity_params['temp_c']])
        axes[0, 0].set_title('Peak Load by Temperature')
        axes[0, 0].set_xlabel('Temperature (°C)')
        axes[0, 0].set_ylabel('Peak Load (kW)')
        axes[0, 0].set_xticklabels(sensitivity_params['temp_c'])
        
        # Total energy by PEV type
        pev_energy = df.groupby('pev_type')['total_energy'].mean()
        axes[0, 1].bar(pev_energy.index, pev_energy.values)
        axes[0, 1].set_title('Average Total Energy by PEV Type')
        axes[0, 1].set_ylabel('Total Energy (kWh)')
        
        # Peak hour distribution
        axes[1, 0].hist(df['peak_hour'], bins=range(25), alpha=0.7, edgecolor='black')
        axes[1, 0].set_title('Distribution of Peak Load Hours')
        axes[1, 0].set_xlabel('Hour of Day')
        axes[1, 0].set_ylabel('Frequency')
        
        # Correlation heatmap (numeric columns only)
        numeric_df = df.select_dtypes(include=[np.number])
        if len(numeric_df.columns) > 1:
            corr_matrix = numeric_df.corr()
            sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, ax=axes[1, 1])
            axes[1, 1].set_title('Parameter Correlations')
        
        plt.tight_layout()
        plt.show()
        
        # Print summary statistics
        print("\n📈 Sensitivity Analysis Summary:")
        print(f"Peak Load Range: {df['peak_load'].min():.1f} - {df['peak_load'].max():.1f} kW")
        print(f"Total Energy Range: {df['total_energy'].min():.1f} - {df['total_energy'].max():.1f} kWh")
        print(f"Most Common Peak Hour: {df['peak_hour'].mode().iloc[0]}:00")
        
        return df
    else:
        print("❌ No successful results for sensitivity analysis")
        return None

# Run sensitivity analysis (limited requests due to API rate limits)
sensitivity_df = sensitivity_analysis(API_KEY, max_requests=16)

## 💾 Data Export and Summary

Let's save our results and create a final summary:

In [ ]:
# Save results to CSV if we have sensitivity data
if sensitivity_df is not None:
    output_file = f"evi_pro_lite_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    sensitivity_df.to_csv(output_file, index=False)
    print(f"💾 Results saved to: {output_file}")
    
    # Display final summary table
    print("\n📋 Final Results Summary:")
    print("=" * 60)
    print(sensitivity_df.round(2).to_string(index=False))

# Create summary of all scenarios tested
print("\n🎯 Demo Summary:")
print("=" * 40)
print(f"✅ API connectivity: {'Success' if test_response else 'Failed'}")
print(f"🌡️ Temperature analysis: {len(temp_results) if 'temp_results' in locals() else 0} scenarios")
print(f"🚗 PEV type analysis: {len(pev_results) if 'pev_results' in locals() else 0} scenarios")
print(f"🔌 Charging behavior analysis: {len(charging_results) if 'charging_results' in locals() else 0} scenarios")
print(f"🌴 California scenarios: {len(ca_results) if 'ca_results' in locals() else 0} scenarios")
print(f"🔬 Sensitivity analysis: {len(sensitivity_df) if sensitivity_df is not None else 0} combinations")

print("\n📚 Next Steps:")
print("- Get your own API key at https://developer.nrel.gov/signup/ for unlimited access")
print("- Explore the full GitHub repository at https://github.com/NREL/EVI-Pro-Lite")
print("- Integrate EV charging profiles with your solar + storage analysis")
print("- Consider time-of-use rates and grid impacts in California")

## 🔗 Integration with Your Solar Analysis

Here's how you could integrate EV charging data with your solar + storage cost analysis:

In [ ]:
def integrate_ev_solar_analysis():
    """
    Example of how to integrate EV charging profiles with solar + storage analysis
    """
    print("🔌⚡ Integration Ideas for Your Cost Analysis:")
    print("=" * 50)
    
    integration_steps = [
        "1. 🚗 Add EV charging load to existing household electricity profiles",
        "2. 📊 Update SAM model inputs to include EV load in sizing solar + storage",
        "3. 💰 Calculate additional electricity costs from EV charging",
        "4. 🔋 Analyze how EV charging affects battery storage utilization",
        "5. ⏰ Consider TOU rates impact on EV charging vs. solar production",
        "6. 🗺️ Map optimal EV + solar scenarios across California counties",
        "7. 💵 Calculate payback periods including EV charging savings"
    ]
    
    for step in integration_steps:
        print(f"   {step}")
    
    print("\n🔧 Code Integration Pattern:")
    print("""
    # In your step6_build_electric_vehicle_load_profiles.py:
    def get_ev_charging_profile(county, scenario_params):
        # Use EVI-Pro Lite API to get EV charging profile
        response = make_api_request(API_KEY, **scenario_params)
        return response['outputs']['load_profile_kw']
    
    # In your step7_combine_real_and_simulated_electricity_loads.py:
    def add_ev_load_to_profile(existing_profile, ev_profile, ev_adoption_rate):
        # Scale EV profile by adoption rate and add to existing load
        scaled_ev = [load * ev_adoption_rate for load in ev_profile]
        combined = [existing + ev for existing, ev in zip(existing_profile, scaled_ev)]
        return combined
    """)

integrate_ev_solar_analysis()

## 📝 Conclusion

This notebook has demonstrated:

✅ **API Connectivity**: Successfully connected to NREL's EVI-Pro Lite API  
✅ **Parameter Exploration**: Tested all available API parameters  
✅ **Scenario Analysis**: Created California-specific EV charging scenarios  
✅ **Sensitivity Analysis**: Analyzed how parameters affect charging patterns  
✅ **Data Visualization**: Created comprehensive plots of charging load profiles  
✅ **Integration Planning**: Outlined how to integrate with your solar cost analysis  

### 🚀 Key Insights:
- **Temperature** significantly affects EV charging energy requirements
- **Vehicle type** (BEV vs PHEV, range) impacts charging patterns
- **Charging behavior** settings can shift peak loads to different hours
- **California scenarios** show regional variation in charging needs

### 📈 Business Value:
This EV charging data can enhance your residential electrification cost analysis by:
- More accurate solar + storage sizing including EV loads
- Better understanding of grid impacts and utility rate implications
- Comprehensive payback period calculations for full electrification
- Regional optimization strategies for California's diverse climate zones

**Next step**: Integrate these EV charging profiles into your `cost_service.py` pipeline!